**Setup**

In [9]:
import os
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

# Set Hugging Face token safely from Colab Secrets or environment
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if hf_token:
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
    print("DuckDB Hugging Face secret configured.")
else:
    print("Warning: HF_TOKEN not found. Set it in Colab Secrets as HF_TOKEN.")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{WAREHOUSE}/fact_content_daily_performance"
DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"

DuckDB Hugging Face secret configured.


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfaheem8988/FlyRank-AI-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Grain: One row = one unique content item (content_hash) evaluated at the close of an observation month.**

* **Feature Observation Window:** Mid-panel month month=2026-03 (March 1, 2026 to March 31, 2026).
* **Outcome / Target Window:** The subsequent month month=2026-04 (April 1, 2026 to April 30, 2026).
* **Target Definition:** Binary indicator ($y \in \{0, 1\}$) where 1 represents an article that had organic click gains or search position improvement in the outcome window (clicks_next_month > clicks_obs_month), and 0 otherwise.
* **Notice on sealed holdout:** The final month (month=2026-06) is reserved strictly as an unseen test month and is never touched during feature and label development.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify observation month date bounds and sample grain
df_window = con.execute(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS distinct_days,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(*) AS total_daily_records
    FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
""").df()

df_window

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,distinct_days,distinct_content_items,total_daily_records
0,2026-03-01,2026-03-31,31,331437,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
### Field Classification for this Lane

#### 1. Features (Available at decision moment March 31, 2026)

* `total_clicks`: Aggregated search clicks (`SUM(gsc_clicks)`) accumulated during March 2026. Knowable at cutoff because GSC performance logs have completed for the month.
* `total_impressions`: Aggregated search impressions (`SUM(gsc_impressions)`) during March 2026. Knowable at cutoff from monthly search visibility.
* `avg_position`: Impression-weighted position (`SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0)`) in March 2026. Knowable at cutoff as the true average rank.
* `ctr`: Click-through rate calculated as `SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0)`. Knowable at cutoff as historical user engagement efficiency.
* `organic_sessions`: Total GA4 organic visits (`SUM(sessions_organic)`) accumulated during March 2026. Knowable at cutoff via on-site analytics sync.

#### 2. Label (Derived strictly from April 2026 outcome window)

* `target_rebound`: Binary flag ($1$ if `april_clicks > march_clicks`, else $0$). Computed strictly from post-cutoff data to measure performance trajectory.

#### 3. Context / Key Identifiers

* `content_hash_id`: Anonymized primary identifier for the content item / URL.
* `client_hash_id`: Anonymized client account identifier.
* `month`: Partition key defining the observation snapshot (`2026-03`).

#### 4. Excluded Fields & Rationale

* **Post-March 31 daily signals (`gsc_clicks`, `ga4_sessions` from April onwards)**: Deliberately excluded from features to prevent lookahead target leakage.
* **Low-volume/inactive records (`gsc_data_available IS FALSE` or `gsc_impressions < 10`)**: Deliberately excluded because low-impression tail queries produce extreme Poisson noise in CTR and average rank calculations without providing actionable editorial recovery headroom.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify schema and non-null presence of planned fields in the March 2026 slice
df_fields = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(content_hash_id) AS non_null_content,
        SUM(gsc_clicks) AS sum_clicks,
        SUM(gsc_impressions) AS sum_impressions,
        SUM(sessions_organic) AS sum_organic_sessions,
        COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_available_count
    FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
""").df()

df_fields

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,non_null_content,sum_clicks,sum_impressions,sum_organic_sessions,gsc_available_count
0,9841378,9841378,821832.0,280657589.0,585408.0,3611061


## 3. Verify it with queries (grain, counts, missing values, windows)



Every claim above gets a query cell here. A contract claim without a query next to it is a guess.

### Verification Queries, Feature Engineering & Leakage Experiment

#### 1. Contract Verifications
* **Fact 1 (Grain)**: Verifies that grouping by `content_hash_id` creates a unique row per content item (`max_rows_per_id = 1`).
* **Fact 2 (Date Span & Row Count)**: Proves exact coverage from `2026-03-01` to `2026-03-31` (31 distinct days) and records total daily observations.
* **Fact 3 (Availability via `IS TRUE`)**: Confirms how many rows survive the contract filter `gsc_data_available IS TRUE` and volume threshold `impressions >= 10 IS TRUE`.

#### 2. Five Feature Frame ("Knowable at the decision moment because...")
* `total_clicks`: Knowable at decision cutoff (2026-03-31) because it aggregates historical daily Search Console click logs up to March 31.
* `total_impressions`: Knowable at decision cutoff because Search Console impression aggregation is finalized for March.
* `avg_position`: Knowable at decision cutoff as the impression-weighted average ranking across all March queries.
* `ctr`: Knowable at decision cutoff because it is derived purely from March clicks divided by impressions.
* `organic_sessions`: Knowable at decision cutoff because GA4 session attribution logs have synced through March 31.

#### 3. The Leakage Trap
* We deliberately add `april_clicks` (a post-cutoff outcome metric) into the feature matrix.
* We observe the artificial ROC-AUC score jump toward near-perfection.
* We immediately delete the leaked column, preserving only the honest pre-cutoff baseline.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# ==============================================================================
# 1. THREE VERIFICATION QUERIES
# ==============================================================================

# FACT 1: Verify Grain (1 row per content_hash_id after aggregation)
query_grain = con.execute(f"""
    WITH aggregated AS (
        SELECT content_hash_id, COUNT(*) AS days_present
        FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
        GROUP BY content_hash_id
    )
    SELECT
        COUNT(*) AS total_unique_content_rows,
        MAX(days_present) AS max_days_recorded,
        MIN(days_present) AS min_days_recorded
    FROM aggregated
""").df()
print("--- Fact 1: Grain Verification (1 row = 1 unique content_hash_id) ---")
print(query_grain)

# FACT 2: Slice Row Count & Exact Date Span
query_span = con.execute(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(DISTINCT report_date) AS total_days_covered,
        COUNT(*) AS total_daily_raw_rows
    FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
""").df()
print("\n--- Fact 2: Date Span & Total Raw Rows ---")
print(query_span)

# FACT 3: Availability Filter Check using IS TRUE
query_avail = con.execute(f"""
    SELECT
        COUNT(*) AS total_raw_daily_records,
        COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_available_rows,
        COUNT(CASE WHEN (gsc_data_available IS TRUE AND gsc_impressions >= 10 IS TRUE) THEN 1 END) AS eligible_volume_rows,
        ROUND(COUNT(CASE WHEN (gsc_data_available IS TRUE AND gsc_impressions >= 10 IS TRUE) THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_surviving
    FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
""").df()
print("\n--- Fact 3: Availability Filter Surviving Rows (IS TRUE) ---")
print(query_avail)

# ==============================================================================
# 2. BUILD THE 5-FEATURE FRAME + HONEST TARGET
# ==============================================================================

df_model = con.execute(f"""
    WITH march_features AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks) AS total_clicks,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
            SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS ctr,
            SUM(sessions_organic) AS organic_sessions
        FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 10 IS TRUE
    ),
    april_outcome AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet('{FACT_DAILY}/month=2026-04/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        m.total_clicks,
        m.total_impressions,
        m.avg_position,
        m.ctr,
        m.organic_sessions,
        COALESCE(a.april_clicks, 0) AS april_clicks,
        CASE WHEN COALESCE(a.april_clicks, 0) > m.total_clicks THEN 1 ELSE 0 END AS target_rebound
    FROM march_features m
    LEFT JOIN april_outcome a ON m.content_hash_id = a.content_hash_id
""").df().fillna(0)

print(f"\n--- Feature DataFrame Preview (Shape: {df_model.shape}) ---")
print(df_model[['content_hash_id', 'total_clicks', 'total_impressions', 'avg_position', 'ctr', 'organic_sessions', 'target_rebound']].head())

# ==============================================================================
# 3. THE LEAKAGE TRAP EXPERIMENT
# ==============================================================================

feature_cols = ['total_clicks', 'total_impressions', 'avg_position', 'ctr', 'organic_sessions']
X_honest = df_model[feature_cols]
y = df_model['target_rebound']

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)

# Train Honest Baseline Model
clf_honest = LogisticRegression(max_iter=1000)
clf_honest.fit(X_train, y_train)
preds_honest = clf_honest.predict_proba(X_test)[:, 1]
honest_auc = roc_auc_score(y_test, preds_honest)

print(f"\n[HONEST MODEL] Test ROC-AUC (5 Clean Pre-Cutoff Features): {honest_auc:.4f}")

# Train Trap / Leaky Model (Injecting april_clicks from post-cutoff window)
X_leaky = df_model[feature_cols + ['april_clicks']]
X_tr_l, X_te_l, _, _ = train_test_split(X_leaky, y, test_size=0.3, random_state=42)

clf_leaky = LogisticRegression(max_iter=1000)
clf_leaky.fit(X_tr_l, y_train)
preds_leaky = clf_leaky.predict_proba(X_te_l)[:, 1]
leaky_auc = roc_auc_score(y_test, preds_leaky)

print(f"[TRAP MODEL] Test ROC-AUC with Leaked 'april_clicks': {leaky_auc:.4f}")
print(f"Leakage Jump: +{leaky_auc - honest_auc:.4f} (Artificial inflation)")

# Clean up: Delete the leaked column from our dataframe
del df_model['april_clicks']
print("\nLeaked feature 'april_clicks' removed from DataFrame. The data contract remains clean.")

--- Fact 1: Grain Verification (1 row = 1 unique content_hash_id) ---
   total_unique_content_rows  max_days_recorded  min_days_recorded
0                     331437                 31                  1

--- Fact 2: Date Span & Total Raw Rows ---
  start_date   end_date  total_days_covered  total_daily_raw_rows
0 2026-03-01 2026-03-31                  31               9841378

--- Fact 3: Availability Filter Surviving Rows (IS TRUE) ---
   total_raw_daily_records  gsc_available_rows  eligible_volume_rows  \
0                  9841378             3611061               2147529   

   pct_surviving  
0          21.82  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Feature DataFrame Preview (Shape: (143206, 8)) ---
            content_hash_id  total_clicks  total_impressions  avg_position  \
0  content_92bc8dfb830d0ade           0.0               13.0     26.846154   
1  content_f1c085e5ea530266           0.0              207.0      5.449275   
2  content_c4901b51a12b1c3b           1.0              218.0      7.793578   
3  content_e68b30ce53db783f           1.0              150.0      4.093333   
4  content_dd37ba6019bb33f7           0.0               45.0      7.355556   

        ctr  organic_sessions  target_rebound  
0  0.000000               0.0               0  
1  0.000000               0.0               0  
2  0.004587               0.0               0  
3  0.006667               0.0               0  
4  0.000000               0.0               0  

[HONEST MODEL] Test ROC-AUC (5 Clean Pre-Cutoff Features): 0.6120
[TRAP MODEL] Test ROC-AUC with Leaked 'april_clicks': 1.0000
Leakage Jump: +0.3880 (Artificial inflation)

Leaked featur

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits & Inherent Slice Constraints

1. **GSC Tail Privacy Thresholding (Underreported Long-Tail Queries)**:
   Google Search Console suppresses very low-volume search queries to protect user privacy. Consequently, long-tail clicks and impressions are aggregated into unsegmented totals or omitted entirely, undercounting the true breadth of early-stage discovery keywords.

2. **Unobserved Macro Influences & Algorithmic Volatility**:
   The warehouse records search metrics (`gsc_clicks`, `gsc_impressions`, `gsc_avg_position`) and on-site behavior (`ga4_sessions`), but cannot measure external macro shifts. Broad core Google algorithm updates, competitor publishing bursts, SERP feature restructurings (such as AI Overviews displacing organic ranks), or server downtime cannot be inferred from observational performance tables alone.

3. **Window Overlap & Observational Bias**:
   Because real-world editorial refreshes happen asynchronously, monthly bucketed slices (`month=2026-03` vs. `month=2026-04`) do not reflect a clean A/B testing intervention. Any post-period uplift captures a mix of content freshness, seasonality, and indexing lag.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantify data limits: long-tail skew and zero-inflation across the catalog
skew_analysis = con.execute(f"""
    SELECT
        COUNT(*) AS total_daily_records,
        COUNT(CASE WHEN gsc_clicks = 0 THEN 1 END) AS zero_click_records,
        ROUND(COUNT(CASE WHEN gsc_clicks = 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_zero_clicks,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY gsc_impressions) AS median_daily_impressions,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY gsc_impressions) AS p95_daily_impressions,
        MAX(gsc_impressions) AS max_daily_impressions
    FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
""").df()

print("--- Data Skew & Zero-Inflation Analysis ---")
skew_analysis

--- Data Skew & Zero-Inflation Analysis ---


,total_daily_records,zero_click_records,pct_zero_clicks,median_daily_impressions,p95_daily_impressions,max_daily_impressions
0,9841378,9423397,95.75,0.0,135.0,40084


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.